In [ ]:
import os

# Chemin du notebook → racine du projet
repo_root = os.path.dirname(os.path.abspath("__file__"))
requirements = os.path.join(repo_root, "requirements.txt")

%pip install -r "{requirements}"

# Mistral OCR

In [ ]:
import base64
import os
from mistralai.client import Mistral
from dotenv import load_dotenv
import pypdfium2 as pdfium
import string

In [ ]:
def extract_native_text_from_pdf(pdf_path: str) -> str:
    pdf = pdfium.PdfDocument(pdf_path)
    page = pdf[0]  # load a page

    # Load a text page helper
    textpage = page.get_textpage()

    # Extract text from the whole page
    return textpage.get_text_bounded()

In [ ]:
def encode_pdf(pdf_path):
    with open(pdf_path, "rb") as pdf_file:
        return base64.b64encode(pdf_file.read()).decode('utf-8')

def ocr_from_pdf(pdf_path: str, client) -> str:
    base64_pdf = encode_pdf(pdf_path)

    ocr_response = client.ocr.process(
        model="mistral-ocr-latest",
        document={
            "type": "document_url",
            "document_url": f"data:application/pdf;base64,{base64_pdf}"
        },
        table_format="html", # default is None
        # extract_header=True, # default is False
        # extract_footer=True, # default is False
        include_image_base64=True
    )
    
    return ocr_response

In [ ]:
def extract_text_from_pdf(pdf_path: str, client) -> str:
    result = extract_native_text_from_pdf(pdf_path)
    
    # On ne fallback en OCR que si on a pas réussi à extraire de text
    if len(result) < 100 :
        print(f"Native extraction from {pdf_path} is impossible, fallback on OCR")
        result = ocr_from_pdf(pdf_path, client)
            
    return result

# Text to File Name

In [ ]:
def sanitize_filename(text):
    # Caractères autorisés : alphanumériques, -, _, ., espace, ()
    allowed = string.ascii_letters + string.digits + "-_.() "
    # Remplace tout caractère non autorisé par '_'
    safe = "".join(c if c in allowed else "_" for c in text)
    # Nettoyage final
    safe = safe.replace(" ", "_").replace("__", "_").strip("_")
    return safe

In [ ]:
def text_to_file_name(pdf_text: str, client) -> str:
    with open("prompt.txt", "r", encoding="utf-8") as f:
        prompt = f.read()
    
    chat_response = client.chat.complete(
        model = "mistral-small-latest",
        messages = [
            {
                "role": "system",
                "content": prompt,
            },
            {
                "role": "user",
                "content": str(pdf_text),
            },
        ]
    )
    
    file_name = chat_response.choices[0].message.content
    
    return sanitize_filename(file_name)

# Rename Invoice

# Full Process

In [ ]:
def rename_file(pdf_path : str, folder_path :str, new_pdf_name : str) :
    """ Renome le fichier en place, en déduplicant les noms si le fichiers existe déjà"""
    counter = 1
    
    new_pdf_path = os.path.join(folder_path, f"{new_pdf_name}.pdf")
    
    while os.path.exists(new_pdf_path):
        new_pdf_path = os.path.join(folder_path, f"{new_pdf_name}({counter}).pdf")
        counter += 1
        
    try :
        os.rename(pdf_path, new_pdf_path)
        print(f"File {os.path.basename(pdf_path)} renamed to {new_pdf_name}.pdf")
    except :
        print(f"ERROR : Impossible to reanme {os.path.basename(pdf_path)} to {new_pdf_name}.pdf")

In [ ]:
def process_single_pdf(folder_path, pdf_name, client):
    print(f"Processing file: {pdf_name}")
    pdf_path = os.path.join(folder_path, pdf_name)
    
    pdf_text = extract_text_from_pdf(pdf_path, client)
    
    # c'est toujours le même nom de fichier qui est rnvoyé
        
    new_pdf_name = text_to_file_name(pdf_text, client)
        
    # Rename file inplace
    rename_file(pdf_path, folder_path, new_pdf_name)

In [ ]:
def process_folder(folder_path):
    load_dotenv()
    api_key = os.getenv("MISTRAL_API_KEY")
    client = Mistral(api_key=api_key)
    
    for file in os.listdir(folder_path):
        if file.endswith(".pdf"):
            process_single_pdf(folder_path, file, client)


In [ ]:
process_folder("./Samples")